# W8T41JZK0ZMEP

Packages

In [ ]:
# Imports
from foodcast.imports import *
os.chdir(find_project_root())
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, _, _, _, DATA_DIR_3_7 = DATA_DIR_3_x

# Settings
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = 'W8T41JZK0ZMEP'
df_uncleaned = load_single_restaurant(loc_id)
df_targeted = pd.read_parquet(DATA_DIR_3_7 / f'{loc_id}.parquet')

In [ ]:
# Remapping was manual, but stored in a yaml now to stay organized
labeling_path = Path('scripts') / 'labeling'
remapping_path = labeling_path / 'remapping' / 'loc6_remappings.yaml'
with open(remapping_path, "r", encoding="utf-8") as f:
    remapping = yaml.load(f, Loader=yaml.FullLoader)

# Extract the parts of the yaml to do the programmatic relabeling
df_relabeled = fully_relabel_and_consolidate(
    df_uncleaned,
    remove=remapping.get("merch_list", []),
    modification_name_changes=remapping.get("modification_name_changes", []),
    vegan_list=remapping.get("vegan_list", []),
    vegetarian_list=remapping.get("vegetarian_list", []),
    meat_list=remapping.get("meat_list", []),
    drinks_list=remapping.get("non_alcoholic_drinks", []),
    alcohol_list=remapping.get("alcoholic_drinks", []),
    merch=remapping.get("merch_list", []),
    rare=remapping.get("rare_list", []),
    unknown=remapping.get("unknown_list", [])
)

df_relabeled.to_parquet(DATA_DIR_3_1 / (loc_id + '_sales_and_menu.parquet'))

df_consolidated = df_relabeled.pipe(rename_items, name_changes={})

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, top_n=30)

df_consolidated.to_parquet(DATA_DIR_3_2 / (loc_id + '_sales_and_menu.parquet'))

In [ ]:
items_less_than_2_dollars = (
    df_targeted
    .groupby('item_name')
    ['unit_price']
    .max()
    .to_frame(name='max_unit_price')
    .query('max_unit_price < 200.0')
    .reset_index()
    .query('~item_name.isin(["Pumpkin Bread","Muffin Marionberry Cobbler","Muffin Chocolate"])')
    .item_name
    .tolist()
    )

missed_merch = []
missed_drinks = []
cookies = []
take_and_go = []
name_changes = {}

modification_name_changes = []

filtered = (
    df_targeted
    .query('~item_name.isin(@items_less_than_2_dollars)')
    .query('~item_name.isin(@missed_merch)')
    .query('~item_name.isin(@missed_drinks)')
    .query('~item_name.isin(@take_and_go)')
    .query('~item_name.isin(@cookies)')
    .pipe(lambda df: rename_items(df, name_changes))
    #.pipe(lambda df: rename_items_by_modifications(df, modification_name_changes))
    )


print(
    filtered
    .groupby('item_name')
    ['item_modifications']
    .apply(lambda s: s.value_counts().index.str.slice(0,20).tolist()[0:3])
    .loc[filtered.item_name.value_counts().index.tolist()]
    .to_frame()
    .join(filtered.item_name.value_counts())
    .reset_index()
    .set_index('count')
    [['item_name','item_modifications']]
    #.iloc[10:,:]
    .to_string()
)

plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

In [ ]:
s = pd.Series([np.nan, 1, np.nan, np.nan, np.nan, 1, np.nan, np.nan, 1, np.nan])
print(s.tolist())
print(strict_bridge_fill(s.to_frame(), limit=2).squeeze().tolist())
print(strict_bridge_fill(s.to_frame(), limit=3).squeeze().tolist())

presence_dict = {}
for item, group in filtered.groupby("item_name"):
    presence_dict[item] = infer_active_days(group["created_at"], max_gap_days=120)
presence_df = pd.concat(presence_dict, axis=1).fillna(False)

presence_weekly = presence_df.resample('W').max().fillna(False).astype(bool).to_period('W')
dish_order = list(filtered.item_name.value_counts().index)
plot_boolean_time_series(presence_weekly, loc_id, before_after_details_true, dish_order, [])
plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

presence_daily = strict_bridge_fill(presence_df, limit=7).resample('D').max()
#presence_daily = (presence_weekly.astype(float).replace(0.0, np.nan)[vegetarian_dishes].resample('D').interpolate(limit=6).fillna(0).astype(int).sum(axis=1).plot())
menu = pd.read_csv(Path("scripts") / "labeling" / "dish_labels" / (loc_id + ".csv"))

vegan_dishes = menu.loc[menu["vegan"], "item_name"]
vegetarian_dishes = menu.loc[menu["vegetarian"], "item_name"]
mpbamod_dishes = menu.loc[menu["mpbamod"], "item_name"]

vegan_dishes_count = presence_daily[vegan_dishes].sum(axis=1)
vegetarian_dishes_count = presence_daily[vegetarian_dishes].sum(axis=1)
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
indicator = pd.to_datetime(promo_date) < presence_daily.index
mpbamod_dishes_count = presence_daily[mpbamod_dishes].sum(axis=1) * indicator
menu_counts = pd.concat([vegan_dishes_count, vegetarian_dishes_count, mpbamod_dishes_count], axis=1).set_axis(['vegan_dishes_count', 'vegetarian_dishes_count', 'mpbamod_dishes_count'], axis=1)
menu_counts.plot()
menu_counts.to_csv(Path("scripts") / "labeling" / "dish_counts" / (loc_id + ".csv"))

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Vegan")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Vegan")')['item_quantity'].sum())

# Your original time series plots
plot_time_series_subset(
    df_uncleaned.query('item_name.str.contains("Vegan Breakfast Sandwich")'), 
    exposure=promo_date,
    freq='D', 
    truncate=False)
plot_time_series_subset(
    df_uncleaned.query('item_name.str.contains("Vegan Breakfast Sandwich")'), 
    exposure=promo_date,
    freq='D',
    truncate=True)
plt.show()

# Parse promo strings / lists
promo_name = before_after_details_true.loc[loc_id,'promo_name']
if isinstance(promo_name, str) and promo_name.startswith('['):
    import ast
    promo_list = ast.literal_eval(promo_name)
    vegan_str, bacon_str = promo_list[0], promo_list[1]
else:
    vegan_str, bacon_str = "Vegan Breakfast Sandwich", "Vegan Cheese"

# Define filters
filters = {
    'Vegan Breakfast Sandwich': lambda df: df['item_name'].str.contains("Vegan Breakfast Sandwich", na=False),
    'Vegan Cheese': lambda df: df['item_modifications'].str.contains("Vegan Cheese", na=False),
    'Unchicken': lambda df: df['item_modifications'].str.contains("Unchicken|Un'chicken", na=False),
    'Vegan Cupcake': lambda df: df['item_modifications'].str.contains("Vegan Cupcake", na=False),
    'Vegan Sausage': lambda df: df['item_modifications'].str.contains("Vegan Sausage", na=False),
    'Vegan Cheese (Alt)': lambda df: df['item_modifications'].str.contains("Vegan Cheese|Vegan Cheddar", na=False)
}

# Plot results
plt.figure(figsize=(12, 6))
for label, filter_func in filters.items():
    data = df_uncleaned.loc[filter_func(df_uncleaned)]
    plt.plot(data.resample('W')['item_quantity'].sum(), label=label)
plt.axvline(x=promo_date, color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
plt.legend()
plt.xticks(rotation=50)
plt.show()

plot_dish_time_series(
    df_consolidated
    .dropna(subset='item_modifications')
    .query('item_name.str.contains("Desserts") and item_modifications.str.contains("Vegan")')
    .assign(item_name = lambda df: df['item_modifications']),
    loc_id,
    before_after_details_true,
    top_n=30)

plot_time_series(
    df_uncleaned, 
    exposure = promo_date,
    freq='W', 
    truncate=False)
plot_time_series(
    df_uncleaned.dropna(subset='item_modifications').query('item_name.str.contains("V - Smash Wrap")'), 
    exposure = promo_date,
    freq='W', 
    truncate=False)
plt.show()


In [ ]:
#df_uncleaned.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Cupcake")').index[0]
#df_uncleaned.loc[:pd.Timestamp('2020-04-27 13:28:58-0400', tz='America/New_York')]

print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Cupcake")').index[0])
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Gf Cake Slice")').index[0])
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Vegan Cheesecake")').index[3])
print(df_consolidated.dropna(subset='item_modifications').query('item_modifications.str.contains("Bar, Vegan")').index[0])
print(df_consolidated.dropna(subset='item_modifications').query('item_name.str.contains("Vegan Breakfast Sandwich")').index[0])

"""vegan cupcake 2020-04-27
vegan cheesecake 2020-02-17
vegan gf cake slice 2020-07-13
vegan raspberry lemon no bake 2021-02-05"""

start_offset = df_uncleaned.index[0] + pd.DateOffset(days=30)
mask = df_uncleaned.item_modifications.value_counts().gt(10) & df_uncleaned.item_modifications.value_counts().index.str.contains("V:|VG|V/|V\)|V\]|V |V\s*$|Vegan|Plant-based|Plant based", case=False, regex=True)
df_uncleaned.reset_index().groupby('item_modifications')['created_at'].first().loc[mask].to_frame(name = 'first').query('@start_offset < first').sort_values('first').join(df_uncleaned.item_modifications.value_counts(), how='left')

df_uncleaned.query('item_name.str.contains("Cheesecake")')[['item_name','item_modifications']].value_counts()
start_offset = df_uncleaned.index[0] + pd.DateOffset(days=30)
mask = df_uncleaned.item_name.value_counts().gt(10) & df_uncleaned.item_name.value_counts().index.str.contains("V:|VG|V/|V\)|V\]|V |V\s*$|Vegan|Plant-based|Plant based", case=False, regex=True)
df_uncleaned.reset_index().groupby('item_name')['created_at'].first().loc[mask].to_frame(name = 'first').query('@start_offset < first').sort_values('first').join(df_uncleaned.item_name.value_counts(), how='left')

df_uncleaned.assign(item_modifications = lambda df: df.item_modifications.fillna('')).query('item_name.str.contains("Raspberry")')#[['item_name','item_modifications']].value_counts()

# clean_df.query('item_name.str.contains("Smash")')[['item_modifications']].value_counts()
# clean_df.query('item_name == "Wraps"')[['item_name','item_modifications','is_plant_based']].value_counts()